In [ ]:
# COLAB SETUP

%cd /content
!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive, runtime
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

/content
Cloning into '/content/proto-tsrl'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 180 (delta 71), reused 93 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 888.26 KiB | 8.30 MiB/s, done.
Resolving deltas: 100% (71/71), done.
/content/proto-tsrl
Mounted at /content/drive
/content/proto-tsrl


In [ ]:
import functools

import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

from src.experiments.ppg.ppg_model import PPGModel
from src.utils.sampling_utils import *
from src.utils.training_utils import *
from ppg_data_utils import *

In [ ]:
# SETTINGS

SEED = 42

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

# data
INCLUDE_CLEAN_DATA = True
INCLUDE_SEMINOISY_DATA = True
INCLUDE_NOISY_DATA = False
TRAIN_SET_SIZE = int(1e5)

# dataloaders
BATCH_SIZE = 256
VAL_RATIO = 0.05
N_WORKERS = 4

# contrastive sampling
# PPG data has length 800, rescaled to [0, 1]
COLLATE_FN_KWARGS = {
    'min_len': 100,
    'max_len': 300,
    'num_neg': 5,
    'min_overlap': 0.3,
    'min_var': 1e-4,
    'max_var': 1e-3
}

# schedule
N_EPOCHS_STAGE1 = 5
N_EPOCHS_STAGE2 = 5
N_EPOCHS_STAGE3 = 5
TOTAL_EPOCHS = N_EPOCHS_STAGE1 + N_EPOCHS_STAGE2 + N_EPOCHS_STAGE3

# logging
CHECKPOINT_EPOCHS = [*range(1, TOTAL_EPOCHS + 1)] # 1-indexed
SAVE_DIR = "/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg/checkpoints"

# architecture
MASK_PROB = 0.2
MID_WEIGHT = 0.5
PROTO_NEG_MARGIN = 0.1
PROTO_DIVERSITY_THRESHOLD = 0.2
LAMBDA_PROTO = 1.0
TEMPERATURE = 1.0
LAMBDA_REPR = 1.0
GRADIENT_CLIP = None

# parameter settings
REP_DIMS = [10, 50, 100]
MODELS = []
CKPT_PATHS = []
for dim in REP_DIMS:
    MODEL = PPGModel(representation_dimension = dim, mask_probability = MASK_PROB)
    if torch.cuda.device_count() > 1:
        MODEL = nn.DataParallel(MODEL)
    elif not isinstance(MODEL, nn.DataParallel):
        MODEL = nn.DataParallel(MODEL)

    ckpt_path = f"{SAVE_DIR}/dim{dim}"

    MODELS.append(MODEL)
    CKPT_PATHS.append(ckpt_path)

# optimizer
OPTIMIZERS = {
    "stage1": torch.optim.Adam(MODEL.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage2": torch.optim.Adam(MODEL.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage3": torch.optim.Adam(MODEL.parameters(), lr = 1e-4, weight_decay = 1e-5),
}

SCHEDULER = None

cuda:0


In [ ]:
# LOAD DATA

X_train, y_train = load_data(
    file_path = '/content/drive/MyDrive/Duke/Senior Year/Thesis/ppg_data',
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'train',
    return_labels = True
)

indices = np.random.default_rng(SEED).permutation(X_train.shape[0])
X_train, y_train = X_train[indices][:TRAIN_SET_SIZE], y_train[indices][:TRAIN_SET_SIZE]

print(f'X_train shape: {X_train.shape}')
print(f'Train set positive samples: {np.sum(y_train)}')

dl_train, dl_val = make_dataloaders(
    X_train,
    y_train,
    batch_size = BATCH_SIZE,
    val_ratio = VAL_RATIO,
    seed = SEED,
    num_workers = N_WORKERS,
    collate_fn_kwargs = COLLATE_FN_KWARGS
)

X_train shape: (100000, 800, 1)
Train set positive samples: 36118
X_test shape: (5278, 800, 1)
Test set positive samples: 747


In [ ]:
### TRAINING

for i, dim in enumerate(REP_DIMS):
    print(f'TRAINING MODEL WITH REPRESENTATION DIMENSION {dim}')
    history = run_training(
        model = MODELS[i],
        train_loader = dl_train,
        val_loader = dl_val,
        device = DEVICE,
        optimizer_dict = OPTIMIZERS,
        epochs_stage1 = N_EPOCHS_STAGE1,
        epochs_stage2 = N_EPOCHS_STAGE2,
        epochs_stage3 = N_EPOCHS_STAGE3,
        scheduler_dict = SCHEDULER,
        mid_weight = MID_WEIGHT,
        proto_neg_margin = PROTO_NEG_MARGIN,
        proto_diversity_threshold = PROTO_DIVERSITY_THRESHOLD,
        lambda_proto = LAMBDA_PROTO,
        temperature = TEMPERATURE,
        lambda_repr = LAMBDA_REPR,
        grad_clip_norm = GRADIENT_CLIP,
        checkpoint_path = CKPT_PATHS[i],
        checkpoint_epochs = CHECKPOINT_EPOCHS,
        collector_fn = None,
        use_amp = True
    )

TRAINING MODEL WITH REPRESENTATION DIMENSION 10
Training for 15 epochs.

=== stage1 (5 epochs) ===
[stage1] epoch 1/5 | global 1/15
  train loss: 3.332541 | val loss: 3.310323
Saved checkpoint at epoch 1
[stage1] epoch 2/5 | global 2/15
  train loss: 3.329146 | val loss: 3.335487
Saved checkpoint at epoch 2
[stage1] epoch 3/5 | global 3/15
  train loss: 3.333297 | val loss: 3.339379
Saved checkpoint at epoch 3
[stage1] epoch 4/5 | global 4/15
  train loss: 3.334338 | val loss: 3.344449
Saved checkpoint at epoch 4
[stage1] epoch 5/5 | global 5/15
  train loss: 3.333762 | val loss: 3.332224
Saved checkpoint at epoch 5

=== stage2 (5 epochs) ===
[stage2] epoch 1/5 | global 6/15
  train loss: 46.369728 | val loss: 46.337676
Saved checkpoint at epoch 6
[stage2] epoch 2/5 | global 7/15
  train loss: 46.368945 | val loss: 46.353589
Saved checkpoint at epoch 7
[stage2] epoch 3/5 | global 8/15
  train loss: 46.370010 | val loss: 46.330591
Saved checkpoint at epoch 8
[stage2] epoch 4/5 | global 

In [ ]:
runtime.unassign()